# Chapter 16 &mdash; $P$-time and NP-time Defined via DTMs and NDTMs

**Concept 3 of the Chapter 16 decomposition:** *$P$-time and NP-time Defined via DTMs and NDTMs*

$P$: steps along a DTM's single path; NP: the <i>maximum</i> steps along any NDTM path.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-P-And-NP-Via-Machines/Concept-P-And-NP-Via-Machines.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The machine definitions, stated carefully because the NP one has a trap.

* $L \in P$ if some **DTM** decides $L$ in $O(n^k)$ steps for some constant $k$. One
  path, count its length.
* $L \in NP$ if some **NDTM** decides $L$ in $O(n^k)$ steps, where the cost of an NDTM
  is the **length of the longest path in its computation tree** &mdash; *not* the total
  number of nodes.

That is the trap. An NDTM's tree can have exponentially many nodes while every path is
short. NP-time measures **depth**, not **work**.

Equivalently (Concept 4): guess a polynomial-length certificate, then check it
deterministically in polynomial time. The guess is the branching; the check is the
path.

## 2. Definitions

### A computation tree, and its two measures

In [ ]:
def tree_stats(branching, depth):
    nodes = sum(branching ** d for d in range(depth + 1))
    return dict(longest_path=depth, total_nodes=nodes)

### A nondeterministic search, and its deterministic simulation

In [ ]:
from itertools import product
def nd_subset_sum(nums, target):
    # ONE path of the NDTM: guess a subset, then add it up.
    # Path length is O(n); the TREE has 2^n leaves.
    return dict(path_length=len(nums), tree_leaves=2 ** len(nums))

def det_subset_sum(nums, target):
    steps = 0
    for bits in product([0, 1], repeat=len(nums)):
        steps += len(nums)
        if sum(n for n, b in zip(nums, bits) if b) == target:
            return True, steps
    return False, steps

## 3. Tests

**Depth versus nodes.** The tree explodes; the paths do not.

In [ ]:
print("%-8s %-16s %s" % ("depth", "longest path", "total nodes"))
for d in [4, 8, 16, 30]:
    s = tree_stats(2, d)
    print("%-8d %-16d %s" % (d, s['longest_path'], format(s['total_nodes'], ',')))
s = tree_stats(2, 30)
assert s['longest_path'] == 30 and s['total_nodes'] > 10 ** 9
print("\nNP-time for that machine is 30, not two billion.")

Subset sum: the NDTM path is linear, the deterministic search is exponential.

In [ ]:
nums = [3, 34, 4, 12, 5, 2]
print("nondeterministic :", nd_subset_sum(nums, 9))
ok, steps = det_subset_sum(nums, 9)
print("deterministic    : found=%s after %d elementary steps" % (ok, steps))
assert ok

The gap grows with $n$.

In [ ]:
print("%-6s %-14s %s" % ("n", "NDTM path", "DTM steps (worst case)"))
for n in [4, 8, 12, 16]:
    nums = list(range(1, n + 1))
    target = sum(nums) + 1                      # unreachable: forces full search
    _, steps = det_subset_sum(nums, target)
    print("%-6d %-14d %s" % (n, n, format(steps, ',')))

So the definitions, side by side.

In [ ]:
print("P  : exists DTM  M, constant k, with M deciding L in O(n^k) STEPS")
print("NP : exists NDTM N, constant k, with N deciding L and every path")
print("     in its computation tree of length O(n^k)")
print()
print("The NDTM is allowed exponentially many paths.  It is not allowed a")
print("single LONG one.")

And why simulation costs exponential time but not exponential *depth*.

In [ ]:
for n in [10, 20, 30]:
    print("  n=%2d : NDTM depth %2d, DTM simulation visits up to %s nodes"
          % (n, n, format(2 ** n, ',')))
print("\nThat is the best known general simulation -- and improving it")
print("to polynomial would prove P = NP.")

## 4. Exercises


1. Why is it wrong to define NP-time as the number of nodes in the tree?
2. Show $P \subseteq NP$ from these definitions.
3. What is the best known deterministic simulation of an NDTM? What does it cost?

In [ ]:
# Your work for the exercises above.